In [0]:
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 13.0827,
    "longitude": 80.2707,
    "start_date": "2026-07-01",
    "end_date": "2026-07-02",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "weather_code",
        "wind_speed_10m"
    ],
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=30)

response.raise_for_status()

data = response.json()

print("API status:", response.status_code)
print("Available fields:", list(data["hourly"].keys()))
print("Number of timestamps:", len(data["hourly"]["time"]))

In [0]:
from pyspark.sql import Row

# Convert API data into rows
rows = [
    Row(
        time=data["hourly"]["time"][i],
        temperature_2m=data["hourly"]["temperature_2m"][i],
        relative_humidity_2m=data["hourly"]["relative_humidity_2m"][i],
        precipitation=data["hourly"]["precipitation"][i],
        rain=data["hourly"]["rain"][i],
        weather_code=data["hourly"]["weather_code"][i],
        wind_speed_10m=data["hourly"]["wind_speed_10m"][i]
    )
    for i in range(len(data["hourly"]["time"]))
]

# Create Spark DataFrame
df = spark.createDataFrame(rows)

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

display(df)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_pipeline.bronze.weather_test")

In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather_test")

print("Bronze rows:", bronze_df.count())
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import col, to_timestamp

# Read Bronze
bronze_df = spark.table("my_pipeline.bronze.weather_test")

# Convert time to timestamp
silver_df = bronze_df.withColumn(
    "time", to_timestamp(col("time"))
)

# Remove duplicate timestamps
silver_df = silver_df.dropDuplicates(["time"])

# Remove rows with missing timestamps
silver_df = silver_df.filter(col("time").isNotNull())

# Display the cleaned data
print("Silver rows:", silver_df.count())
silver_df.printSchema()
display(silver_df)

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_pipeline.silver.weather_test")

In [0]:
silver_check = spark.table("my_pipeline.silver.weather_test")

print("Silver rows:", silver_check.count())
silver_check.printSchema()

In [0]:
tables = [
    "my_pipeline.bronze.weather",
    "my_pipeline.silver.weather",
    "my_pipeline.gold.weather"
]

for table in tables:
    df = spark.table(table)
    print(f"{table}: {df.count()} rows")

In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather")

print("Total rows:", bronze_df.count())
print("Unique timestamps:",
      bronze_df.select("time").distinct().count())

In [0]:
silver_df = spark.table("my_pipeline.silver.weather")

print("Total rows:", silver_df.count())
print("Unique timestamps:",
      silver_df.select("time").distinct().count())

silver_df.printSchema()

In [0]:
gold_df = spark.table("my_pipeline.gold.weather")

print("Total rows:", gold_df.count())
print("Unique dates:",
      gold_df.select("date").distinct().count())

gold_df.printSchema()

In [0]:
import requests

url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 13.0827,
    "longitude": 80.2707,
    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "wind_speed_10m,"
        "weather_code,"
        "rain,"
        "apparent_temperature,"
        "precipitation,"
        "surface_pressure,"
        "cloud_cover,"
        "wind_direction_10m,"
        "wind_gusts_10m"
    ),
    "forecast_days": 16,
    "timezone": "Asia/Kolkata"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()
print("API status:", response.status_code)
print("Available hourly records:", len(data["hourly"]["time"]))

In [0]:
print(type(data))
print(type(data["hourly"]))
print(type(data["hourly"]["time"]))

In [0]:
existing_times = set(
    str(row["time"])[:16]
    for row in spark.table("my_pipeline.bronze.weather")
        .select("time")
        .collect()
)

new_times = set(t[:16] for t in data["hourly"]["time"])

print("Fetched:", len(data["hourly"]["time"]))
print("New timestamps:", len(new_times - existing_times))


In [0]:
import pandas as pd

hourly = data["hourly"]

new_pdf = pd.DataFrame(hourly)
new_pdf["location"] = "Chennai"
new_pdf["latitude"] = data["latitude"]
new_pdf["longitude"] = data["longitude"]
new_pdf["source"] = "Open-Meteo"

new_df = spark.createDataFrame(new_pdf)

print("Prepared rows:", new_df.count())
new_df.printSchema()

In [0]:
bronze_df = spark.table("my_pipeline.bronze.weather")

print("Existing Bronze rows:", bronze_df.count())
bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

new_df = new_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

new_df = new_df.select(*bronze_df.columns)

print("New rows:", new_df.count())
new_df.printSchema()

In [0]:
new_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("my_pipeline.bronze.weather")

In [0]:
print(
    "Total Bronze rows:",
    spark.table("my_pipeline.bronze.weather").count()
)

In [0]:
print("Bronze:", spark.table("my_pipeline.bronze.weather").count())
print("Silver:", spark.table("my_pipeline.silver.weather").count())
print("Gold:", spark.table("my_pipeline.gold.weather").count())